# LinguaRoute
I081 Dhruv Paleja · I053 Manan Shah  
Natural Language Processing — Dr. Ami Munshi

Aim: identify the language of written content and suggest a language desk for review.

## 1. Data
Use 18 languages from WiLI-2018. The official test split stays separate. The training script downloads the archive and checks the publisher's checksum.

In [1]:
from pathlib import Path
import json
import contextlib
import io
from training import train, LANGUAGES

# Fit and evaluate; show a compact summary instead of the full training log.
with contextlib.redirect_stdout(io.StringIO()):
    result = train()
print('Languages:', len(LANGUAGES))
print('Training rows:', result['split']['train_used'])
print('Test rows:', result['split']['official_test_subset'])
print('Features:', result['features'])

Languages: 18
Training rows: 8975
Test rows: 9000
Features: 40000


## 2. Features
Keep accents and scripts. Character 2–4 grams are converted to TF-IDF vectors. This example shows why whole-word vocabularies are not required.

In [2]:
from training import normalize, grams
text = 'Bonjour à tous'
print(normalize(text))
print(grams(text)[:18])

bonjour à tous
['bo', 'on', 'nj', 'jo', 'ou', 'ur', 'r ', ' à', 'à ', ' t', 'to', 'ou', 'us', 'bon', 'onj', 'njo', 'jou', 'our']


## 3. Model comparison
Validation macro F1 chooses the SVM regularization value. Naive Bayes provides a simple baseline. These results are for held-out Wikipedia paragraphs, not every possible message.

In [3]:
print('Selected C:', result['selected_c'])
for name, key in [('Naive Bayes', 'baseline'), ('Linear SVM', 'test')]:
    scores = result[key]
    print(f"{name:14} accuracy={scores['accuracy']:.4f}  macro F1={scores['macro_f1']:.4f}")

Selected C: 2.0
Naive Bayes    accuracy=0.9754  macro F1=0.9765
Linear SVM     accuracy=0.9880  macro F1=0.9881


## 4. Look at every language
Precision measures how often a predicted language was right. Recall measures how many true examples of that language were found.

In [4]:
report = json.loads(Path('evaluation/classification-report.json').read_text())
print(f"{'Language':14} {'Precision':>10} {'Recall':>8} {'F1':>8}")
for label, name in sorted(LANGUAGES.items(), key=lambda item: item[1]):
    row = report[label]
    print(f"{name:14} {row['precision']:10.3f} {row['recall']:8.3f} {row['f1-score']:8.3f}")

Language        Precision   Recall       F1
Arabic              1.000    0.998    0.999
Bengali             0.990    0.952    0.970
Danish              0.998    0.982    0.990
Dutch               0.998    0.994    0.996
English             0.895    0.992    0.941
Finnish             0.998    0.996    0.997
French              0.988    0.994    0.991
German              0.970    0.986    0.978
Hindi               0.998    0.978    0.988
Indonesian          0.996    0.990    0.993
Italian             0.984    0.996    0.990
Marathi             0.998    0.976    0.987
Polish              0.996    0.994    0.995
Portuguese          0.990    0.992    0.991
Spanish             0.998    0.982    0.990
Swedish             0.998    1.000    0.999
Turkish             1.000    0.994    0.997
Vietnamese          1.000    0.988    0.994


## 5. Try a new sentence
The exported model uses the same TF-IDF calculation in Python and the browser. Decision scores are not probabilities.

In [5]:
from inference import predict
examples = [
    'Our team would like more information about the international conference.',
    'Notre équipe souhaite recevoir davantage d’informations sur la conférence.',
    'हमारी टीम अंतरराष्ट्रीय सम्मेलन के बारे में अधिक जानकारी प्राप्त करना चाहती है।'
]
for text in examples:
    output = predict(text)
    print(text)
    print([(r['language'], round(r['score'], 3)) for r in output])
    print()

Our team would like more information about the international conference.
[('English', 0.109), ('Portuguese', -0.796), ('Marathi', -0.807)]

Notre équipe souhaite recevoir davantage d’informations sur la conférence.
[('French', 0.226), ('Hindi', -0.826), ('Portuguese', -0.826)]



हमारी टीम अंतरराष्ट्रीय सम्मेलन के बारे में अधिक जानकारी प्राप्त करना चाहती है।
[('Hindi', 0.684), ('Marathi', -0.751), ('Vietnamese', -0.99)]



## 6. Review mistakes
An error can come from a short paragraph, shared vocabulary, a related language or a noisy source label. Read a few mistakes before treating a score as certainty.

In [6]:
errors = json.loads(Path('evaluation/errors.json').read_text(encoding='utf-8'))
print('Misclassified test paragraphs:', len(errors))
for row in errors[:3]:
    print('Actual:', LANGUAGES[row['actual']], '| Predicted:', LANGUAGES[row['predicted']])
    print(row['text'][:160], '\n')
print('Review policy:', result['review_policy'])
print('Coverage:', round(result['selective_test']['coverage'], 4))

Misclassified test paragraphs: 108
Actual: Danish | Predicted: German
Alt Günding, Bergkirchen, Bergkirchen-Lus, Bibereck, Breitenau, Deutenhausen, Eisolzried, Eschenried, Facha, Feldgeding, Gröbenried, Heißhof, Hopfenau, Kienaden 

Actual: Indonesian | Predicted: English
Tieszen, Richard, Mathematics, in David Smith and Barry Smith, eds., The Cambridge Companion to Husserl (Cambridge University Press, circa 2005). 

Actual: Portuguese | Predicted: German
Pax, Ferdinand Albin & Hoffmann, Käthe (1924): Euphorbiaceae-Crotonoideae-Acalypheae-Acalyphinae. In: Engler, Adolf: Das Pflanzenreich Series IV '147.XVI (85):  

Review policy: {'score_floor': -0.25, 'margin_floor': 0.25, 'min_letters': 30, 'min_feature_coverage': 0.08}
Coverage: 0.99


## Conclusion
Character patterns provide a small, inspectable language classifier. The application adds batch routing, explanations and a sentence map. Its limits are important: unsupported languages, transliteration and within-sentence code-switching need separate evaluation.

Source: Martin Thoma, [WiLI-2018](https://doi.org/10.5281/zenodo.841984), ODbL. See DATA_SOURCES.md for attribution and exact data handling.